### Run in Collab using GPU runtime

In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

In [8]:
df=pd.read_csv('merged_data.csv', index_col='Date', parse_dates=True)
threshold = len(df) * 0.7
df = df.dropna(thresh=threshold, axis=1)
df = df.ffill().bfill()
returns = df.pct_change().dropna()
scaler = StandardScaler()
scaled_returns = scaler.fit_transform(returns)
data_tensor = torch.tensor(scaled_returns.T, dtype=torch.float32)
stock_names = list(returns.columns)
num_stocks = len(stock_names)

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 1. SETUP & DATA LOADING
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# CHANGE THIS to your filename in Colab
file_name = 'merged_data.csv'

if not os.path.exists(file_name):
    print(f"ERROR: {file_name} not found. Please upload it to Colab side-bar.")
else:
    # Load and preprocess
    df = pd.read_csv(file_name, index_col='Date', parse_dates=True)
    returns = df.pct_change().dropna()

    scaler = StandardScaler()
    scaled_returns = scaler.fit_transform(returns)

    # [Stocks, Time]
    data_tensor = torch.tensor(scaled_returns.T, dtype=torch.float32)
    stock_names = list(returns.columns)
    num_stocks = len(stock_names)
    lookback = 20

    # 2. GENERATE SLIDING WINDOWS (Fix for the NameError)
    X_train = []
    Y_train = []
    for i in range(data_tensor.shape[1] - lookback):
        X_train.append(data_tensor[:, i:i+lookback])
        Y_train.append(data_tensor[:, i+lookback])

    # 3. MODEL ARCHITECTURE
    class GraphDiscoveryNet(nn.Module):
        def __init__(self, num_nodes, lookback):
            super(GraphDiscoveryNet, self).__init__()
            self.adj_weights = nn.Parameter(torch.randn(num_nodes, num_nodes))
            self.gcn_layer = nn.Linear(lookback, 64)
            self.output_layer = nn.Linear(64, 1)

        def forward(self, x):
            adj = torch.tanh(self.adj_weights)
            adj = adj + torch.eye(adj.size(0)).to(device)
            support = torch.relu(self.gcn_layer(x))
            output = torch.matmul(adj, support)
            prediction = self.output_layer(output)
            return prediction, adj

    # 4. INITIALIZATION
    model = GraphDiscoveryNet(num_stocks, lookback).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    criterion = nn.MSELoss()
    l1_lambda = 1e-4

    # Move lists to device
    X_train_tensors = [t.to(device) for t in X_train]
    Y_train_tensors = [t.to(device) for t in Y_train]

    # 5. TRAINING LOOP
    print("Starting training...")
    for epoch in range(101):
        model.train()
        total_loss = 0
        for i in range(len(X_train_tensors)):
            optimizer.zero_grad()
            pred, _ = model(X_train_tensors[i])
            mse_loss = criterion(pred.squeeze(), Y_train_tensors[i])
            l1_penalty = l1_lambda * torch.norm(model.adj_weights, 1)
            loss = mse_loss + l1_penalty
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 20 == 0:
            print(f"Epoch {epoch} | Avg Loss: {total_loss/len(X_train):.6f}")

    # 6. METRICS FOR PRESENTATION
    model.eval()
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for i in range(len(X_train_tensors)):
            pred, _ = model(X_train_tensors[i])
            all_preds.append(pred.squeeze().cpu().numpy())
            all_actuals.append(Y_train_tensors[i].cpu().numpy())

    all_preds, all_actuals = np.array(all_preds), np.array(all_actuals)
    mse = mean_squared_error(all_actuals, all_preds)
    hit_ratio = np.mean(np.sign(all_preds) == np.sign(all_actuals))

    print("\n" + "="*30)
    print("      GNN TRAINING METRICS      ")
    print("="*30)
    print(f"MSE: {mse:.6f}")
    print(f"Directional Hit Ratio: {hit_ratio:.2%}")
    print("="*30)

    # 7. PAIR EXTRACTION & SAVING
    with torch.no_grad():
        _, final_adj = model(X_train_tensors[-1])
        adj_matrix = (final_adj - torch.eye(num_stocks).to(device)).cpu().numpy()

    pairs = []
    for i in range(num_stocks):
        for j in range(num_stocks):
            if i != j:
                pairs.append({'Stock_A': stock_names[i], 'Stock_B': stock_names[j], 'Strength': adj_matrix[i, j]})

    pairs_df = pd.DataFrame(pairs).sort_values(by='Strength', ascending=False)

    # Define and create models directory
    # Note: In Colab, '../models' refers to the root. Using './models' is safer.
    model_dir = './models'
    if not os.path.exists(model_dir):
        os.makedirs(model_dir)

    pairs_df.head(20).to_csv(f'{model_dir}/discovered_positive_pairs.csv', index=False)
    pairs_df.tail(20).to_csv(f'{model_dir}/discovered_negative_pairs.csv', index=False)
    torch.save(model.state_dict(), f'{model_dir}/gnn_pairs_discovery.pth')

    print(f"\nDiscovery Complete. Files saved to: {os.path.abspath(model_dir)}")

Using device: cuda
Starting training...


/tmp/ipykernel_3047/3625439125.py:22: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = df.pct_change().dropna()


Epoch 0 | Avg Loss: 1.082036
Epoch 20 | Avg Loss: 0.879604
Epoch 40 | Avg Loss: 0.811316
Epoch 60 | Avg Loss: 0.788075
Epoch 80 | Avg Loss: 0.764694
Epoch 100 | Avg Loss: 0.771305

      GNN TRAINING METRICS      
MSE: 0.776784
Directional Hit Ratio: 66.01%

✅ Discovery Complete. Files saved to: /content/models
